In [1]:
# 交叉熵
import numpy as np

def numpy_cross_entropy(logits, target):
    """
    logits: 网络的原始输出 (N, C)，也就是batchsize， d_model，是logits
    target: 真实类别的索引 (N,) 例如 [0, 2, 1]
    """
    # 1. 计算 Softmax (为了数值稳定性，减去每行的最大值)
    exps = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    softmax_probs = exps / np.sum(exps, axis=1, keepdims=True)
    
    # 2. 提取对应真实类别的预测概率
    n = logits.shape[0]
    # 使用高级索引获取对应 target 的概率值
    correct_logprobs = -np.log(softmax_probs[range(n), target] + 1e-12)
    
    # 3. 计算平均损失
    loss = np.sum(correct_logprobs) / n
    return loss

# 测试
logits = np.array([[2.0, 1.0, 0.1], [1.0, 3.0, 0.2]])
target = np.array([0, 1])
print(f"NumPy Loss: {numpy_cross_entropy(logits, target):.4f}")

import torch
import torch.nn.functional as F

def torch_cross_entropy(logits, target):
    """
    logits: (N, C)
    target: (N,) 类别索引
    """
    # 1. 计算 LogSoftmax (比先 Softmax 再 Log 更数值稳定)
    # 公式: log_softmax(x) = x - log(sum(exp(x)))
    log_probs = logits - torch.log(torch.exp(logits).sum(dim=1, keepdim=True))
    
    # 2. 按照 target 索引提取对应的 log_probs
    # 使用 gather 函数：在维度 1 上根据 target 索引取值
    n = logits.shape[0]
    gathered_log_probs = log_probs.gather(1, target.unsqueeze(1))
    
    # 3. 取负号并求均值
    loss = -gathered_log_probs.mean()
    return loss

# 测试与官方对比
logits = torch.tensor([[2.0, 1.0, 0.1], [1.0, 3.0, 0.2]], requires_grad=True)
target = torch.tensor([0, 1])

my_loss = torch_cross_entropy(logits, target)
official_loss = F.cross_entropy(logits, target)

print(f"Manual Torch Loss: {my_loss.item():.4f}")
print(f"Official Torch Loss: {official_loss.item():.4f}")

NumPy Loss: 0.2981
Manual Torch Loss: 0.2981
Official Torch Loss: 0.2981


In [2]:
# Kmeans
import numpy as np

def kmeans(X, k, max_iters=100, tol=1e-4):
    """
    X: 输入数据 (n_samples, n_features)
    k: 分类数
    max_iters: 最大迭代次数
    tol: 收敛阈值（质心变化小于该值则停止）
    """
    n_samples, n_features = X.shape
    
    # 1. 初始化质心：从数据集中随机选 k 个点
    # 使用 choice 确保不选到重复的点
    indices = np.random.choice(n_samples, k, replace=False)
    centroids = X[indices]
    
    for i in range(max_iters):
        # 2. 分配步骤 (Assign)
        # 计算距离：利用广播机制计算 (n_samples, 1, n_features) - (k, n_features)
        # 得到形状 (n_samples, k) 的距离矩阵
        distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
        
        # 找到每个样本距离最近的质心索引
        labels = np.argmin(distances, axis=1)
        
        # 3. 更新步骤 (Update)
        new_centroids = np.array([
            X[labels == j].mean(axis=0) if len(X[labels == j]) > 0 else centroids[j]
            for j in range(k)
        ])
        
        # 4. 检查收敛：如果质心变化很小，提前退出
        if np.linalg.norm(new_centroids - centroids) < tol:
            print(f"Converged at iteration {i}")
            break
            
        centroids = new_centroids
        
    return centroids, labels

# --- 测试代码 ---
if __name__ == "__main__":
    # 生成简单的模拟数据：两个中心点附近的数据
    data = np.vstack([
        np.random.randn(50, 2) + np.array([5, 5]),
        np.random.randn(50, 2) + np.array([-5, -5])
    ])
    
    centers, labels = kmeans(data, k=2)
    print("Final Centroids:\n", centers)

Converged at iteration 1
Final Centroids:
 [[ 5.1925907   5.24217306]
 [-5.23498637 -4.93293557]]


In [ ]:
# AUC
def AUC(y_score, y_true):
    # y_score: 预测分数；y_true：真实标签 0/1
    # 按照标签升序排列
    indices = sorted(range(len(y_score)), key = lambda i: y_score[i])
    sorted_labels = [y_true[i] for i in indices]
    pos_c = np.sum(y_true)
    neg_c = len(y_true) - pos_c
    rank_sum = 0
    for i in range(len(sorted_labels)):
        if sorted_labels[i] == 1:
            rank_sum += (i+1)
    auc = (rank_sum - pos_c * (pos_c + 1) / 2) / (pos_c * neg_c)
    return auc

# softmax
import numpy as np
def softmax_basic(x):
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x)

def softmax_stable(x):
    max_x = np.max(x, axis=-1, keepdims=True)
    exp_x = np.exp(x-max_x)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

In [1]:
# LayerNorm
import torch
import torch.nn as nn
import math

class LayerNormFast(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        # x: [..., dim] 最后一层归一化
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - mean)/torch.sqrt(var+self.eps)
        return self.gamma * x_norm + self.beta

In [ ]:
import torch
import torch.nn as nn

class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5, elementwise_affine=True):
        """
        参数：
            normalized_shape：需要归一的维度
            eps：小常数
            elementwise_affine：是否使用科学系的gamma和beta
        """
        super(LayerNorm, self).__init__()
        if isinstance(normalized_shape,int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        if self.elementwise_affine:
            self.gamma = nn.Parameter(torch.ones(self.normalized_shape))
            self.beta = nn.Parameter(torch.zeros(self.normalized_shape))
        else:
            self.gamma = None
            self.beta = None
    
    def forward(self, x):
        assert x.dim() >= len(self.normalized_shape), \
            f"输入维度 {x.dim()} 小于归一化维度 {len(self.normalized_shape)}"
        dims = list(range(-len(self.normalized_shape),0)) # [-2,-1]
        mean = x.mean(dims, keepdim=True)
        var = x.var(dims, keepdim=True, unbiased=False)
        std = torch.sqrt(var + self.eps)
        x_normalized = (x-mean)/std
        if self.elementwise_affine:
            x_out = self.gamma * x_normalized + self.beta
        else:
            x_out = x_normalized
        return x_out
    

class RMSNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5, elementwise_affine=True):
        super(RMSNorm,self).__init__()
        if isinstance(normalized_shape,int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        if self.elementwise_affine:
            self.gamma = nn.Parameter(torch.ones(self.normalized_shape))
        else:
            self.gamma = None
        
    def forward(self, x):
        assert x.dim() >= len(self.normalized_shape)
        dims = list(range(-(len(self.normalized_shape)),0))
        rms = torch.sqrt(x.pow(2).mean(dims, keepdim=True) + self.eps)
        if self.elementwise_affine:
            return self.gamma * (x/rms)
        return x/rms

# 示例输入
x = torch.randn(2,3,4)
layer_norm = LayerNorm(x.shape[-1])
rms_norm = RMSNorm(x.shape[-1])
out = layer_norm(x)
print(x)
print(out)
print(rms_norm(x))

tensor([[[-0.3837,  1.4554,  0.4561, -0.8562],
         [ 0.6142, -1.5578,  0.8507, -0.3781],
         [ 0.3791, -0.7226, -1.1410, -1.3359]],

        [[ 0.9136, -0.3118,  0.8885,  0.9752],
         [-1.9619, -2.2943,  1.9423,  0.4023],
         [-0.4022,  0.4103, -0.8498, -1.1665]]])
tensor([[[-0.6272,  1.4639,  0.3277, -1.1645],
         [ 0.7699, -1.5148,  1.0187, -0.2738],
         [ 1.6328, -0.0264, -0.6564, -0.9500]],

        [[ 0.5538, -1.7290,  0.5069,  0.6684],
         [-0.8520, -1.0429,  1.3896,  0.5054],
         [ 0.1685,  1.5395, -0.5868, -1.1212]]], grad_fn=<AddBackward0>)
tensor([[[-0.4285,  1.6255,  0.5095, -0.9563],
         [ 0.6411, -1.6262,  0.8880, -0.3947],
         [ 0.3914, -0.7461, -1.1780, -1.3793]],

        [[ 1.1178, -0.3815,  1.0870,  1.1931],
         [-1.0863, -1.2704,  1.0754,  0.2227],
         [-0.5178,  0.5282, -1.0941, -1.5019]]], grad_fn=<MulBackward0>)


In [3]:
# layernorm
import torch
import torch.nn as nn

class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5, elementwise_affine=True):
        """
        参数：
            normalized_shape：需要归一的维度
            eps：小常熟
            elementwise_affine：是否使用科学系的gamma和beta
        """
        super(LayerNorm, self).__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.elementwise_affine = elementwise_affine

        if self.elementwise_affine:
            self.gemma = nn.Parameter(torch.ones(normalized_shape))
            self.beta = nn.Parameter(torch.zeros(normalized_shape))
        else:
            # self.register_parameter('gamma', None)
            # self.register_parameter('beta', None)
            self.gemma = None
            self.beta = None

    def forward(self, x):
        assert x.dim() >= len(self.normalized_shape), \
            f"输入维度 {x.dim()} 小于归一化维度 {len(self.normalized_shape)}"
        original_shape = x.shape

        dims = list(range(-len(self.normalized_shape), 0))
        mean = x.mean(dim=dims, keepdim=True)
        var = x.var(dim=dims, keepdim=True, unbiased=False)
        std = torch.sqrt(var + self.eps)

        x_normalized = (x-mean)/std
        if self.elementwise_affine:
            x_out = self.gemma * x_normalized + self.beta
        else:
            x_out = x_normalized
        return x_out

# 示例输入
x = torch.randn(2,3,4)
layer_norm = LayerNorm(x.shape[-1])
out = layer_norm(x)
print(x)
print(out)

tensor([[[ 0.6816,  0.9945, -1.2394, -0.5350],
         [-0.9441, -0.5525, -0.8635, -0.7715],
         [-0.6843,  0.9478, -0.9146,  1.4627]],

        [[ 1.3226, -1.7417,  1.0371, -0.5217],
         [-1.1375,  1.3119, -1.9889,  1.4656],
         [ 0.1458,  0.0150, -0.4257,  1.1143]]])
tensor([[[ 0.7806,  1.1265, -1.3428, -0.5643],
         [-1.1011,  1.5737, -0.5505,  0.0779],
         [-0.8681,  0.7289, -1.0934,  1.2327]],

        [[ 1.0491, -1.4266,  0.8184, -0.4409],
         [-0.6968,  0.9282, -1.2616,  1.0302],
         [-0.1184, -0.3511, -1.1350,  1.6045]]], grad_fn=<AddBackward0>)


In [ ]:
# 多头注意力机制（selfattention，支持多头）
import torch
import torch.nn as nn
import math

class SelfAttentionFast(nn.Module):
    def __init__(self, d_model, n_heads=8):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k =  d_model//n_heads
        self.n_heads = n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, mask = None):
        B, L, D = x.shape
        # 分别表征 批次大小，序列长度，特征维度
        Q = self.W_q(x).view(B, L, self.n_heads, self.d_k).transpose(1,2) # shape: B, n_heads, L, d_k
        K = self.W_k(x).view(B, L, self.n_heads, self.d_k).transpose(1,2)
        V = self.W_v(x).view(B, L, self.n_heads, self.d_k).transpose(1,2)

        scores = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(self.d_k) # shape: B, n_heads, L, L
        if mask is not None:
            scores = scores.masked_fill(mask==0, float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        # matmul(attn, V) shape: B, n_heads, L, d_k
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, L, D)  # shape: B, L, n_heads, d_k
        return self.W_o(out)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads=8, dropout=0.1, bias=True):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model//n_heads

        self.W_q = nn.Linear(d_model, d_model, bias=bias)
        self.W_k = nn.Linear(d_model, d_model, bias=bias)
        self.W_v = nn.Linear(d_model, d_model, bias=bias)
        
        self.W_o = nn.Linear(d_model, d_model, bias=bias)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None, is_causal=False):
        # mask: [batch_size, 1, seq_len, seq_len] 或 [batch_size, seq_len, seq_len] 0表示 mask，1 表示保留
        # is_causal: 是否使用因果掩码（上三角为0）

        batch_size, seq_len, _ = x.shape

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        Q = Q.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1,2)
        K = K.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1,2)
        V = V.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1,2)
        
        scores = torch.matmul(Q, K.transpose(-2,-1))//math.sqrt(self.d_k)

        if is_causal:
            causal_mask = torch.triu(
                torch.ones(seq_len, seq_len, device=x.device),
                diagonal=1
            ).bool()
            scores = scores.masked_fill(causal_mask, float('-inf'))

        if mask is not None:
            if mask.dim() == 3:
                mask = mask.unsqueeze(1)
            scores = scores.masked_fill(mask==0, float('-inf'))

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        out = torch.matmul(attn_weights, V)
        out = out.transpose(1,2).contiguous().view(batch_size, seq_len, self.d_model)
        output = self.W_o(out)

        return output, attn_weights

In [ ]:
# Cross Attention
import torch
import torch.nn as nn
import torch.nn.functional as F

class CrossAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        # 定义投影矩阵
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
    def forward(self, x, context, mask=None):
        """
        x: (B, Lq, D) - 比如来自 Decoder 的特征
        context: (B, Lkv, D) - 比如来自 Encoder 的特征或文本 Embedding
        """
        B, Lq, D = x.shape
        Lkv = context.shape[1]
        
        # 1. 线性变换并拆分为多头
        # (B, L, D) -> (B, L, H, d_k) -> (B, H, L, d_k)
        Q = self.W_q(x).view(B, Lq, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(context).view(B, Lkv, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(context).view(B, Lkv, self.n_heads, self.d_k).transpose(1, 2)
        
        # 2. 计算注意力分数 (Scaled Dot-Product)
        # Q: (B, H, Lq, d_k), K^T: (B, H, d_k, Lkv) -> scores: (B, H, Lq, Lkv)
        scores = torch.matmul(Q, K.transpose(-1, -2)) / (self.d_k ** 0.5)
        
        # 3. 如果有 Mask (比如处理 Padding)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
            
        attn = F.softmax(scores, dim=-1)
        
        # 4. 加权求和并合并多头
        # (B, H, Lq, Lkv) * (B, H, Lkv, d_k) -> (B, H, Lq, d_k)
        out = torch.matmul(attn, V)
        
        # 恢复形状: (B, H, Lq, d_k) -> (B, Lq, H*d_k)
        out = out.transpose(1, 2).contiguous().view(B, Lq, D)
        
        # 5. 最后一次线性投影
        return self.W_o(out)

# --- 测试代码 ---
B, D, n_heads = 2, 128, 8
cross_attn = CrossAttention(d_model=D, n_heads=n_heads)

query_seq = torch.randn(B, 10, D)    # 长度为 10 的查询序列
context_seq = torch.randn(B, 20, D)  # 长度为 20 的上下文序列

output = cross_attn(query_seq, context_seq)
print(f"输出形状: {output.shape}") # 应该输出 (2, 10, 128)

In [ ]:
# 带有KV Cache的MHA
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttentionWithCache(nn.Module):
    def __init__(self, d_models, n_heads = 8, dropout=0.1, bias=True):
        super().__init__()
        assert d_models % n_heads == 0
        self.d_k = d_models//n_heads
        self.n_heads = n_heads
        
        self.W_q = nn.Linear(d_models, d_models, bias=bias)
        self.W_k = nn.Linear(d_models, d_models, bias=bias)
        self.W_v = nn.Linear(d_models, d_models, bias=bias)

        self.W_o = nn.Linear(d_models, d_models, bias=bias)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, past_key_value=None, use_cache=False, mask=None):
        B, L, D = x.shape

        Q = self.W_q(x).view(B,L,self.n_heads,self.d_k).transpose(1,2)
        K = self.W_k(x).view(B,L,self.n_heads,self.d_k).transpose(1,2)
        V = self.W_v(x).view(B,L,self.n_heads,self.d_k).transpose(1,2)

        if past_key_value is not None:
            past_k, past_v = past_key_value
            K = torch.cat([past_k, K], dim=2) # batch, n_heads, past_len+seq_len, d_k
            V = torch.cat([past_v, V], dim=2)
        present_key_value = (K, V) if use_cache else None

        score = torch.matmul(Q, K.transpose(-2,-1))/math.sqrt(self.d_k)
        if mask is not None:
            score = score.masked_fill(mask==0, float('-inf'))

        attn = F.softmax(score, dim=-1)
        attn = self.dropout(attn)

        output = torch.matmul(attn, V)
        output = output.transpose(1,2).contiguous().view(B,L,D)

        output = self.W_o(output)
        return output, present_key_value

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class GroupedQueryAttentionWithCache(nn.Module):
    def __init__(self, d_model, n_heads=8, n_kv_heads=None, dropout=0.1, bias=True):
        """
        GQA: Grouped Query Attention with KV Cache
        
        Args:
            d_model: 模型维度
            n_heads: Query头数 (通常n_heads >= n_kv_heads)
            n_kv_heads: Key/Value头数，若为None则等于n_heads（即标准MHA）
                       若为1则等于MQA，若1 < n_kv_heads < n_heads则为GQA
            dropout: dropout概率
            bias: 是否使用偏置
        """
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        # 如果n_kv_heads未指定，默认为n_heads（标准MHA）
        # 如果n_kv_heads=1，则为MQA（Multi-Query Attention）
        # 如果1 < n_kv_heads < n_heads，则为GQA
        self.n_kv_heads = n_kv_heads if n_kv_heads is not None else n_heads
        assert n_heads % self.n_kv_heads == 0, "n_heads必须能被n_kv_heads整除"
        
        # 每个KV头对应的Query头数量
        self.n_rep = n_heads // self.n_kv_heads
        self.d_kv = d_model // self.n_kv_heads  # 每个KV头的维度
        
        # Q投影: d_model -> d_model (n_heads * d_k)
        self.W_q = nn.Linear(d_model, d_model, bias=bias)
        
        # K,V投影: d_model -> n_kv_heads * d_k (注意不是d_model!)
        # 这是GQA的关键：KV的参数量减少为 n_kv_heads/n_heads
        self.W_k = nn.Linear(d_model, self.n_kv_heads * self.d_k, bias=bias)
        self.W_v = nn.Linear(d_model, self.n_kv_heads * self.d_k, bias=bias)
        
        # 输出投影
        self.W_o = nn.Linear(d_model, d_model, bias=bias)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, past_key_value=None, use_cache=False, mask=None):
        """
        Args:
            x: 输入张量 [B, L, D]
            past_key_value: 之前的KV Cache，tuple of (K, V)
                           K/V shape: [B, n_kv_heads, past_len, d_k]
            use_cache: 是否返回KV Cache用于下一步
            mask: 注意力掩码
            
        Returns:
            output: [B, L, D]
            present_key_value: 如果use_cache=True，返回更新后的(K, V)
        """
        B, L, D = x.shape
        
        # 1. 线性投影
        # Q: [B, L, D] -> [B, L, n_heads, d_k] -> [B, n_heads, L, d_k]
        Q = self.W_q(x).view(B, L, self.n_heads, self.d_k).transpose(1, 2)
        
        # K, V: [B, L, D] -> [B, L, n_kv_heads, d_k] -> [B, n_kv_heads, L, d_k]
        # 注意：这里n_kv_heads可能小于n_heads
        K = self.W_k(x).view(B, L, self.n_kv_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, L, self.n_kv_heads, self.d_k).transpose(1, 2)
        
        # 2. 处理KV Cache（推理优化关键）
        if past_key_value is not None:
            past_k, past_v = past_key_value
            # 拼接历史KV和当前KV
            # K: [B, n_kv_heads, past_len, d_k] + [B, n_kv_heads, L, d_k] 
            # -> [B, n_kv_heads, past_len+L, d_k]
            K = torch.cat([past_k, K], dim=2)
            V = torch.cat([past_v, V], dim=2)
        
        # 保存当前的KV用于返回
        present_key_value = (K, V) if use_cache else None
        
        # 3. GQA核心：将K,V重复扩展到与Q相同的头数
        # K, V当前: [B, n_kv_heads, seq_len, d_k]
        # 需要扩展到: [B, n_heads, seq_len, d_k]
        # 通过repeat_interleave实现：每个KV头复制n_rep次
        if self.n_rep > 1:
            # K: [B, n_kv_heads, seq_len, d_k] -> [B, n_kv_heads, 1, seq_len, d_k] 
            #    -> repeat -> [B, n_kv_heads, n_rep, seq_len, d_k]
            #    -> reshape -> [B, n_heads, seq_len, d_k]
            K = K.unsqueeze(2).repeat(1, 1, self.n_rep, 1, 1).reshape(
                B, self.n_heads, K.size(2), self.d_k
            )
            V = V.unsqueeze(2).repeat(1, 1, self.n_rep, 1, 1).reshape(
                B, self.n_heads, V.size(2), self.d_k
            )
        # 如果n_rep=1（MHA），K,V已经是n_heads，无需操作
        # 如果n_kv_heads=1（MQA），则复制n_heads次
        
        # 4. 计算注意力分数
        # Q: [B, n_heads, L, d_k], K: [B, n_heads, seq_len, d_k]
        # scores: [B, n_heads, L, seq_len]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # 5. 应用mask（支持causal mask和padding mask）
        if mask is not None:
            # mask: [B, 1, L, seq_len] 或 [B, 1, 1, seq_len] 等广播形状
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # 6. 加权求和
        # attn_weights: [B, n_heads, L, seq_len]
        # V: [B, n_heads, seq_len, d_k]
        # output: [B, n_heads, L, d_k]
        attn_output = torch.matmul(attn_weights, V)
        
        # 7. 合并多头并输出投影
        # [B, n_heads, L, d_k] -> [B, L, n_heads, d_k] -> [B, L, D]
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, L, self.d_model)
        output = self.W_o(attn_output)
        
        if use_cache:
            return output, present_key_value
        return output


# ==================== 使用示例 ====================

def demo_gqa():
    # 配置
    batch_size = 2
    d_model = 512
    n_heads = 8
    n_kv_heads = 2  # GQA: 4个Query头共享1个KV头 (8/2=4)
    seq_len = 10
    
    # 创建模块
    gqa = GroupedQueryAttentionWithCache(
        d_model=d_model, 
        n_heads=n_heads, 
        n_kv_heads=n_kv_heads
    )
    
    print(f"GQA配置: n_heads={n_heads}, n_kv_heads={n_kv_heads}, n_rep={gqa.n_rep}")
    print(f"KV Cache压缩比: {n_heads}/{n_kv_heads} = {gqa.n_rep}x")
    
    # 模拟自回归生成（step by step）
    print("\n=== 模拟自回归生成 ===")
    
    # 初始输入（prompt）
    prompt_len = 4
    x_prompt = torch.randn(batch_size, prompt_len, d_model)
    
    # 第一步：处理prompt，生成KV Cache
    output, kv_cache = gqa(x_prompt, use_cache=True)
    print(f"Prompt处理: input shape={x_prompt.shape}, output shape={output.shape}")
    print(f"KV Cache K shape: {kv_cache[0].shape}, V shape: {kv_cache[1].shape}")
    
    # 后续每一步只输入一个新token，利用KV Cache
    for step in range(3):
        x_new = torch.randn(batch_size, 1, d_model)  # 每次只输入1个token
        output, kv_cache = gqa(x_new, past_key_value=kv_cache, use_cache=True)
        print(f"Step {step+1}: input shape={x_new.shape}, "
              f"KV Cache长度={kv_cache[0].size(2)}, output shape={output.shape}")
    
    # 对比不同配置的参数量
    print("\n=== 参数量对比 ===")
    configs = [
        ("MHA", 8, 8),
        ("GQA-4", 8, 2),   # 4个query头共享1个kv头
        ("GQA-8", 8, 1),   # 8个query头共享1个kv头 (即MQA)
    ]
    
    for name, nh, nkv in configs:
        m = GroupedQueryAttentionWithCache(d_model, nh, nkv)
        total = sum(p.numel() for p in m.parameters())
        kv_params = m.W_k.weight.numel() + m.W_v.weight.numel()
        print(f"{name}: 总参数量={total:,}, KV投影参数量={kv_params:,}")


def demo_comparison():
    """对比MHA、GQA、MQA的KV Cache大小"""
    d_model = 4096
    n_heads = 32
    batch_size = 1
    seq_len = 2048
    
    configs = [
        ("MHA", 32, 32),
        ("GQA-4 (LLaMA2-70B风格)", 32, 8),
        ("GQA-8 (LLaMA3风格)", 32, 4),
        ("MQA", 32, 1),
    ]
    
    print("KV Cache显存占用对比 (batch=1, seq_len=2048, d_model=4096):")
    print("-" * 60)
    
    for name, nh, nkv in configs:
        # 每个token的KV Cache = 2 * n_kv_heads * d_k * 2 bytes (fp16)
        d_k = d_model // nh
        kv_per_token = 2 * nkv * d_k * 2  # K和V，fp16
        total_kv = kv_per_token * seq_len / (1024**2)  # MB
        
        print(f"{name:25s}: n_kv_heads={nkv:2d}, "
              f"KV Cache={total_kv:.2f} MB "
              f"(压缩比 {nh//nkv}x)")


if __name__ == "__main__":
    demo_gqa()
    print("\n")
    demo_comparison()

In [ ]:
# FlashAttention
import torch
import torch.nn.functional as F
import math
from typing import Optional, Tuple

class FlashAttentionV1:
    def __init__(self, Br: int=128, Bc: int=128):
        self.Br = Br # Query Block size, Block size of Row 表示遍历Query块的行数
        self.Bc = Bc # Key/ Value size, Block size of Column, 表示遍历key/value块的列数

    def forward(self, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, causal:bool=False, scale: Optional[float] = None)-> torch.Tensor:
        B, H, N, D = Q.shape # Batch size, Heads/ n_heads 注意力头数（已经是多头了）, Number of tokens/Sequence Length 序列长度, Dimension/d_head 每一个头的维度
        if scale is None:
            scale = 1.0 / math.sqrt(D)

        O = torch.zeros_like(Q)
        Tr = (N + self.Br - 1) // self.Br # Number of Tile Rows query方向的分块数，Tile划分
        Tc = (N + self.Bc - 1) // self.Bc # Number of Tile Columns key/value方向的分块数， Tile划分

        # V1版本：外层K，V块的循环，串行加载到SRAM
        for j in range(Tc):
            # 加载Kj，Vj到SRAM
            kv_start = j*self.Bc
            kv_end = min(kv_start+self.Bc, N)
            Kj = K[:, :, kv_start:kv_end, :]
            Vj = V[:, :, kv_start:kv_end, :]

            # 内层循环Q块，每个Q块和当前的Kj Vj计算
            for i in range(Tr):
                q_start = i * self.Br
                q_end = min(q_start + self.Br, N)
                Qi = Q[:, :, q_start:q_end, :]

                # 加载Oi，mi，li到SRAM
                Oi = Q[:, :, q_start:q_end, :].clone()
                mi = torch.full((B, H, q_end-q_start), float('-inf'), device=Q.device, dtype=torch.float32)
                li = torch.zeros(B, H, q_end - q_start, device=Q.device, dtype=torch.float32)

                # 计算Sij = Qi @ Kj^T
                Sij = torch.matmul(Qi, Kj.transpose(-2,-1)) * scale

                # 因果掩码
                if causal:
                    q_pos = torch.arange(q_start, q_end, device=Q.device).view(1,1,-1,1)
                    k_pos = torch.arange(kv_start, kv_end, device=Q.device).view(1,1,1,-1)
                    mask = q_pos < k_pos
                    Sij = Sij.masked_fill(mask, float('-inf'))
                
                # Online Softmax
                mij = Sij.max(dim=-1).values
                mij_new = torch.maximum(mi, mij)

                # 缩放并更新
                Pij = torch.exp(Sij - mij_new.unsqueeze(-1))
                Pij = torch.nan_to_num(Pij, nan=0.0)

                lij = Pij.sum(dim=-1)
                li_new = torch.exp(mi - mij_new) * li + lij

                # 更新输出
                Oi = (torch.exp(mi-mij_new).unsqueeze(-1) * Oi + torch.matmul(Pij, Vj))

                # 写回HBM
                Q[:, :, q_start:q_end, :] = Oi / li_new.unsqueeze(-1)

                mi = mij_new
                li = li_new
        return O
    
class FlashAttentionV2:
    def __inif__(self, Br:int=128, Bc:int=128):
        self.Br = Br
        self.Bc = Bc
    def forward(self, Q, K, V, causal: bool=False, scale:Optional[float]=None) -> torch.Tensor:
        B,H,N,D = Q.shape
        if scale is None:
            scale = 1.0/math.sqrt(D)

        O = torch.zeros_like(Q)
        Tr = (N + self.Br - 1) // self.Br
        Tc = (N + self.Bc - 1) // self.Bc

        for i in range(Tr):
            q_start = i*self.Br
            q_end = min(q_start+self.Br, N)
            Qi = Q[:, :, q_start:q_end, :]
            actual_Br = q_end - q_start

            mi = torch.full((B,H,actual_Br), float('-inf'), device=Q.device, dtype=torch.float32)
            li = torch.zeros(B, H, actual_Br, device=Q.device, dtype=torch.float32)
            Oi = torch.zeros(B, H, actual_Br, D, device=Q.device, dtype=Q.dtype)

            for j in range(Tc):
                kv_start = j * self.Bc
                kv_end = min(kv_start + self.Bc, N)
                Kj = K[:, :, kv_start:kv_end, :]
                Vj = V[:, :, kv_start:kv_end, :]
                
                # 计算 Sij
                Sij = torch.matmul(Qi, Kj.transpose(-2, -1)) * scale
                
                # 因果掩码（V2优化：提前跳过无效块）
                if causal:
                    q_pos = torch.arange(q_start, q_end, device=Q.device).view(1, 1, -1, 1)
                    k_pos = torch.arange(kv_start, kv_end, device=Q.device).view(1, 1, 1, -1)
                    
                    # V2优化：如果整个块都在因果掩码之下，直接跳过
                    if kv_end <= q_start:
                        continue  # 整个块都是未来信息，跳过
                    
                    mask = q_pos < k_pos
                    Sij = Sij.masked_fill(mask, float('-inf'))
                
                # Online Softmax（与V1相同数学，但顺序不同）
                mij = Sij.max(dim=-1).values
                mij_new = torch.maximum(mi, mij)
                
                Pij = torch.exp(Sij - mij_new.unsqueeze(-1))
                Pij = torch.nan_to_num(Pij, nan=0.0)
                
                lij = Pij.sum(dim=-1)
                li_new = torch.exp(mi - mij_new) * li + lij
                
                # 更新输出（在SRAM中完成，最后写回一次！）
                Oi = (torch.exp(mi - mij_new).unsqueeze(-1) * Oi + 
                      torch.matmul(Pij, Vj))
                
                mi = mij_new
                li = li_new
            
            # V2关键：每个Q块只写回一次HBM！
            O[:, :, q_start:q_end, :] = Oi / li.unsqueeze(-1)
        
        return O


class FlashAttentionV2Optimized(FlashAttentionV2):
    """
    FlashAttention V2 的进一步优化版本
    
    额外优化：
    1. 因果掩码的块级跳过（V2核心优化）
    2. Split-K for long sequences
    3. 更好的warp级并行
    """
    
    def forward(self, Q, K, V, causal=False, scale=None):
        B, H, N, D = Q.shape
        if scale is None:
            scale = 1.0 / math.sqrt(D)
        
        O = torch.zeros_like(Q)
        Tr = (N + self.Br - 1) // self.Br
        Tc = (N + self.Bc - 1) // self.Bc
        
        for i in range(Tr):
            q_start = i * self.Br
            q_end = min(q_start + self.Br, N)
            Qi = Q[:, :, q_start:q_end, :]
            actual_Br = q_end - q_start
            
            mi = torch.full((B, H, actual_Br), float('-inf'), 
                           device=Q.device, dtype=torch.float32)
            li = torch.zeros(B, H, actual_Br, device=Q.device, dtype=torch.float32)
            Oi = torch.zeros(B, H, actual_Br, D, device=Q.device, dtype=Q.dtype)
            
            # V2关键优化：确定K,V的有效范围
            # 对于causal，只需要处理 k <= q 的块
            kv_start_j = 0 if not causal else max(0, q_start // self.Bc)
            
            for j in range(kv_start_j, Tc):
                kv_start = j * self.Bc
                kv_end = min(kv_start + self.Bc, N)
                Kj = K[:, :, kv_start:kv_end, :]
                Vj = V[:, :, kv_start:kv_end, :]
                
                Sij = torch.matmul(Qi, Kj.transpose(-2, -1)) * scale
                
                if causal:
                    q_pos = torch.arange(q_start, q_end, device=Q.device).view(1, 1, -1, 1)
                    k_pos = torch.arange(kv_start, kv_end, device=Q.device).view(1, 1, 1, -1)
                    mask = q_pos < k_pos
                    Sij = Sij.masked_fill(mask, float('-inf'))
                
                # Online softmax
                mij = Sij.max(dim=-1).values
                mij_new = torch.maximum(mi, mij)
                
                Pij = torch.exp(Sij - mij_new.unsqueeze(-1))
                Pij = torch.nan_to_num(Pij, nan=0.0)
                
                lij = Pij.sum(dim=-1)
                li_new = torch.exp(mi - mij_new) * li + lij
                
                Oi = (torch.exp(mi - mij_new).unsqueeze(-1) * Oi + 
                      torch.matmul(Pij, Vj))
                
                mi = mij_new
                li = li_new
            
            O[:, :, q_start:q_end, :] = Oi / li.unsqueeze(-1)
        
        return O

In [ ]:
# TF-IDF BM25 只是示例，不用掌握

import math
from collections import Counter, defaultdict
import numpy as np

class TFIDF:
    def __init__(self, documents, use_sublinear=False):
        # documents：二维list
        self.documents = documents
        self.N = len(documents)
        self.use_sublinear=use_sublinear

        self.df = defaultdict(int) # document frequency
        self.doc_tfs = [] # 每个文档的词频

        for doc in documents:
            tf = Counter(doc) # 单词哈希表
            self.doc_tfs.append(tf) # 文档的词频就是每个doc的词频的汇总
            for term in tf: # 去重后的单词哈希表
                self.df[term] += 1 # 统计整个documents每一个词的出现次数

        self.idf = {}
        for term, df in self.df.items():
            self.idf[term] = math.log((self.N + 1) / (df + 1)) + 1
    
    def compute_tf(self, term, doc_idx):
        tf = self.doc_tfs[doc_idx].get(term, 0)
        if self.use_sublinear:
            return 1 + math.log(tf) if tf>0 else 0
        else:
            total_terms = sum(self.doc_tfs[doc_idx].values())
            return tf / total_terms if total_terms > 0 else 0
        
    def compute_tfidf(self, term, doc_idx):
        tf = self.compute_tf(term, doc_idx)
        idf = self.idf.get(term, 0)
        return tf * idf
    
    def get_doc_vector(self, doc_idx):
        vector = {}
        for term in self.doc_tfs[doc_idx]:
            vector[term] = self.compute_tfidf(term, doc_idx)
        return vector
    
    def score(self, query, doc_idx):
        score = 0.0
        query_terms = Counter(query)
        for term, qtf in query_terms.items():
            quert_tf = 1+ math.log(qtf) if self.use_sublinear else qtf
            doc_tfidf = self.compute_tfidf(term, doc_idx)
            score += quert_tf * doc_tfidf
        return score
    
    def search(self, query, top_k=10):
        """检索 top-k 文档"""
        scores = [(idx, self.score(query, idx)) for idx in range(self.N)]
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]
    
    def get_tfidf_matrix(self):
        """获取完整的 TF-IDF 矩阵（稀疏）"""
        # 构建词汇表
        vocab = list(self.idf.keys())
        word2idx = {w: i for i, w in enumerate(vocab)}
        
        # 构建矩阵
        matrix = np.zeros((self.N, len(vocab)))
        for d in range(self.N):
            for term, tfidf in self.get_doc_vector(d).items():
                matrix[d, word2idx[term]] = tfidf
        
        return matrix, vocab
    
class BM25:
    def __init__(self, documents, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.documents = documents
        self.N = len(documents)
        
        # 计算文档长度和平均长度
        self.doc_lens = [len(d) for d in documents]
        self.avgdl = sum(self.doc_lens) / self.N
        
        # 构建倒排索引和IDF
        self.doc_freqs = []  # 每个文档的词频
        self.inverted_index = defaultdict(list)  # 词 -> 文档列表
        
        for idx, doc in enumerate(documents):
            freq = Counter(doc)
            self.doc_freqs.append(freq)
            for word in freq:
                self.inverted_index[word].append(idx)
        
        # 预计算IDF
        self.idf = {}
        for word, docs in self.inverted_index.items():
            n_q = len(docs)
            self.idf[word] = math.log((self.N - n_q + 0.5) / (n_q + 0.5) + 1)
    
    def score(self, query, doc_idx):
        """计算单文档得分"""
        score = 0.0
        doc_freq = self.doc_freqs[doc_idx]
        doc_len = self.doc_lens[doc_idx]
        
        for q in query:
            if q not in doc_freq:
                continue
            
            f = doc_freq[q]  # TF
            idf = self.idf.get(q, 0)
            
            # BM25 公式
            numerator = f * (self.k1 + 1)
            denominator = f + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
            
            score += idf * numerator / denominator
        
        return score
    
    def search(self, query, top_k=10):
        """检索 top-k 文档"""
        scores = [(idx, self.score(query, idx)) for idx in range(self.N)]
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]


In [ ]:
# 2026-03-02尝试没有任何提示下手搓带有KV Cache的MHA
import torch
import torch.nn
import math

# 遗忘：当前的K和V是怎么得到的？如果是直接对输入进行映射，那么输入是不是应该只是新的token？
# 定义的时候有没有nn.Module
# dropout是在哪里使用？对O吗？

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads=8, dropout=0.1, bias=True):
        super().__init__()
        assert d_model%n_heads==0
        self.d_k = d_model//n_heads
        self.n_heads = n_heads
        self.bias = bias

        self.W_q = nn.Linear(d_model, d_model, bias=self.bias)
        self.W_k = nn.Linear(d_model, d_model, bias=self.bias)
        self.W_v = nn.Linear(d_model, d_model, bias=self.bias)
        
        self.W_o = nn.Linear(d_model, d_model, bias=self.bias)
        self.dropout = nn.Dropout(dropout)

    # 第一个错误点：遗忘了mask参数
    def forward(self, x, past_key_values=None, use_cache=True, mask=None):
        B, L, D = x.shape
        
        Q = self.W_q(x).view(B,L,self.n_heads, self.d_k).transpose(1,2)
        K = self.W_k(x).view(B,L,self.n_heads, self.d_k).transpose(1,2)
        V = self.W_v(x).view(B,L,self.n_heads, self.d_k).transpose(1,2)

        if past_key_values is not None:
            past_k, past_v = past_key_values
            # K = torch.cat([past_k, K], dim=-1)
            # V = torch.cat([past_v, V], dim=-1)
            # 第二个错误点：拼接的时候dim拼接错误了，拼接到维度上了，实际上应该拼接到seq_len上
            K = torch.cat([past_k, K], dim=2)
            V = torch.cat([past_v, V], dim=2)
        present_key_value = (K,V) if use_cache else None

        score = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(self.d_k)
        # 第三个错误点：遗忘了mask
        if mask is not None:
            score =score.masked_fill(mask==0, float('-inf'))

        # 第四个错误点：命名错了，score是softmax之前的定义，经过softmax之后已经是attn了
        # score = torch.softmax(score, dim=-1)
        attn = torch.softmax(dim=-1)
        # 第五个错误点：dropout的是softmax之后的attn结果，而不是output
        attn = self.dropout(attn)
        # 第六个错误点：最终的结果应该是output
        # attn = torch.matmul(score, V).transpose(1,2).contiguous().view(B,L,D)
        output = torch.matmul(attn,V).transpose(1,2).contiguous().view(B,L,D)
        output = self.W_o(output)
        # output = self.dropout(output)
        return output, present_key_value
    
# 2026-03-02 没有提示下尝试手撕LayerNorm
import torch
import torch.nn as nn
import math

# 第一个错误点：没有nn.Module
class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps = 1e-5, elementwise_affine=True):
        # 第一个错误点
        super().__init__()
        self.eps = eps
        # 第二个错误点，对类型的判断没有加目标类型
        # if isinstance(normalized_shape):
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = normalized_shape
        self.elementwise_affine = elementwise_affine
        if elementwise_affine:
            self.gamma = nn.Parameter(torch.ones(self.normalized_shape))
            self.beta = nn.Parameter(torch.zeros(self.normalized_shape))
        else:
            self.gamma = None
            self.beta = None

    def forward(self, x):
        # 第三个错误点：没有判断输入的维度和归一化维度
        assert x.dim() >= len(self.normalized_shape)
        # 第四个错误点：没有依据归一维度给出x求均值和方差的dim
        dims = list(range(-len(self.normalized_shape), 0))
        # 第五个错误点：dim错误，没有keepdim参数，var没有unbiased参数，var代表方差，没有球根好
        # mean = x.mean(dim=self.normalized_shape)
        # var = x.var(dim = self.normalized_shape)
        # output = (x-mean) / (var + self.eps)
        mean = x.mean(dims, keepdim=True)
        var = x.var(dims, keepdim=True, unbiased=False)
        std = math.sqrt(var + self.eps)
        x_normalized = (x-mean)/std
        if self.elementwise_affine:
            return self.gamma * x_normalized + self.beta
        return x_normalized

In [ ]:
# 2026-03-03无提示下尝试手搓
# 遗忘：RMS如何求均方根
# 遗忘：如何用随机整数或者随机小数（都服从标准正态分布）来初始化参数
# 遗忘：MHA忘记添加bias参数了，并且对score和attn的命名没有搞清楚

import torch
import torch.nn as nn
import math
class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5,elementwise_affine=True):
        super().__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        if self.elementwise_affine:
            self.gamma = nn.Parameter(torch.ones(self.normalized_shape))
            self.beta = nn.Parameter(torch.zeros(self.normalized_shape))
        else:
            self.gamma = None
            self.beta = None

    def forward(self, x):
        assert x.dim() >= len(self.normalized_shape)
        dims = list(range(-len(self.normalized_shape),0))
        mean = x.mean(dim=dims, keepdim=True)
        var = x.var(dim = dims, keepdim=True, unbiased=False)
        std = math.sqrt(var + self.eps)
        x_normalized = (x-mean)/std
        if self.elementwise_affine:
            return self.gamma * x_normalized + self.beta
        return x_normalized
    
class RMSNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5, elementwise_affine=True):
        super().__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        if self.elementwise_affine:
            self.gamma = nn.Parameter(torch.ones(self.normalized_shape))
        else:
            self.gamma = None
    
    def forward(self, x):
        assert x.dim() >= len(self.normalized_shape)
        dims = list(range(-len(self.normalized_shape), 0))
        # 第一个错误点：均方根求错了
        # RMS = sum(x**2).mean() + self.eps
        # x_normalized = x / math.sqrt(RMS)
        RMS = torch.sqrt(x.pow(2).mean(dims, keepdim=True) + self.eps)
        x_normalized = x / RMS
        if self.elementwise_affine:
            return self.gamma * x_normalized
        return x_normalized
    
class MultiHeadAttentionWithKVCache(nn.Module):
    # 第二个错误点：缺少了bias参数
    # def __init__(self, d_models, n_heads = 8, dropout=0.1):
    def __init__(self, d_models, n_heads = 8, dropout=0.1, bias=True):
        super().__init__()
        assert d_models%n_heads==0
        self.d_k = d_models/n_heads
        self.n_heads = n_heads

        # self.W_q = nn.Linear(d_models, d_models)
        # self.W_k = nn.Linear(d_models, d_models)
        # self.W_v = nn.Linear(d_models, d_models)
        
        # self.W_o = nn.Linear(d_models, d_models)
        self.W_q = nn.Linear(d_models, d_models, bias=bias)
        self.W_k = nn.Linear(d_models, d_models, bias=bias)
        self.W_v = nn.Linear(d_models, d_models, bias=bias)
        
        self.W_o = nn.Linear(d_models, d_models, bias=bias)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, past_key_values, use_cache=True, mask = None):
        B, L, D = x.shape
        Q = self.W_q(x).view(B,L,self.n_heads, self.d_k).transpose(1,2)
        K = self.W_k(x).view(B,L,self.n_heads, self.d_k).transpose(1,2)
        V = self.W_v(x).view(B,L,self.n_heads, self.d_k).transpose(1,2)

        if past_key_values is not None:
            past_key, past_value = past_key_values
            K = torch.cat([past_key, K], dim=2)
            V = torch.cat([past_value, V], dim=2)
        if use_cache:
            present_key_values = (K, V)
        else:
            present_key_values = None

        # 第三个错误点：名称以及mask位置错误，softmax内部为score，需要对内部进行掩码，再softmax，而且softmax要指定目标维度
        # score = torch.matmul(Q, K.transpose(-2,-1))
        # if mask is not None:
        #     score = score.masked_fill(mask, float('-inf'))
        # attn = score / math.sqrt(self.d_k)
        # attn = torch.softmax(self.dropout(attn))
        score = torch.matmul(Q,K.transpose(-2,-1))/math.sqrt(self.d_k)
        if mask is not None:
            score = score.masked_fill(mask==0, float('-inf'))
        attn = torch.softmax(score, dim=-1)
        attn = self.dropout(attn)

        output = torch.matmul(attn, V)
        # 第四个错误点：W_o接受的输入是B，L，D，所以在上一部矩阵乘法之后就把维度转换回来
        # output = self.W_o(output).transpose(1,2).contiguous().view(B,L,D)
        output = output.transpose(1,2).contiguous().view(B,L,D)
        output = self.W_o(out)

        return output, present_key_values


In [ ]:
# 2026-03-06 无提示手搓
import torch
import torch.nn as nn
import math
# 疑惑：为什么gamma和beta参数的size都是self.normalized_shape，这个输入是什么？应该是x的维度吗？ 解释：normalized_shape 描述的是形状（例如 (768,)），而 torch.mean 和 torch.var 的 dim 参数需要的是维度索引（整数）
# 疑惑：var里面的unbiased参数的意义是什么？ 解释：表示是否使用无偏估计，var默认unbiased为True，表示计算样本方差，分母是N-1；False表示计算总体方差，分母是N，在归一化中都用总体方差，因此置为False
class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5, elementwise_affine=True):
        super().__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        if self.elementwise_affine:
            self.gamma = nn.Parameter(torch.ones(self.normalized_shape))
            self.beta = nn.Parameter(torch.zeros(self.normalized_shape))
        else:
            self.gamma = None
            self.beta = None
    def forward(self, x):
        assert x.dim() >= len(self.normalized_shape)
        dims = list(range(-len(self.normalized_shape),0))
        mean = x.mean(dim=dims, keepdim=True)
        var = x.var(dim=dims, keepdim=True, unbiased=False)
        std = math.sqrt(var + self.eps)
        x_normalized = (x-mean)/std
        if self.elementwise_affine:
            return self.gamma * x_normalized + self.beta
        return x_normalized
    
class RMSNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5, elementwise_affine=True):
        super().__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        if self.elementwise_affine:
            self.gamma = nn.Parameter(torch.ones(self.normalized_shape))
        else:
            self.gamma = None

    def forward(self, x):
        assert x.dim() >= len(self.normalized_shape)
        dims = list(range(-len(self.normalized_shape),0))
        RMS = x * torch.rsqrt(x.pow(2).mean(dim=dims, keepdim=True) + self.eps)
        if self.elementwise_affine:
            return self.gamma * RMS
        return RMS
# 疑惑：在init函数里面的biased的意义是什么？为什么默认为True 解答：就是线性映射层是否添加偏置b
# 疑惑：torch.concat和torch.cat有什么区别？dropout是在softmax之前还是之后？ 解答：两个没有区别，只是适配了不同的名称习惯；dropout在MHA中一般是对softmax后的结果（attn）和W_o映射后的结果使用
# 疑惑：transpose里面参数-1,-2和-2,-1有没有区别？我倾向于没有，因为就是那两个维度。 解答：没有区别
# 疑惑：softmax有torch和F的实现方式，有什么区别？
# 解答：torch.nn.functional.softmax(input, dim=None)：如果不指定 dim 参数（即保持默认值 None），那么 softmax 会作用于整个张量的所有元素。
# 它会将输入展平为一个一维向量，计算 softmax，然后再恢复为原形状。最终输出的形状与输入相同，但所有元素之和为 1。这种方式较少使用，因为通常我们希望对某个特定维度进行归一化。
# torch.softmax(input, dim)（张量方法或 torch.softmax 函数）：这里的 dim 是必需参数，没有默认值。如果忘记提供 dim，会导致 TypeError，程序无法运行
class MultiHeadAttentionWithKVCache(nn.Module):
    def __init__(self, d_model:int, n_heads:int=8, dropout=0.1, biased=True):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model / n_heads
        self.biased = biased

        self.W_q = nn.Linear(self.d_model, self.d_model, self.biased)
        self.W_k = nn.Linear(self.d_model, self.d_model, self.biased)
        self.W_v = nn.Linear(self.d_model, self.d_model, self.biased)

        self.dropout = nn.Dropout(dropout)
        self.W_o = nn.Linear(self.d_model, self.d_model, self.biased)

    def forward(self, x, past_key_values=None, use_cache=False, mask=None):
        B,L,D = x.shape
        Q = self.W_q(x).view(B,L,self.n_heads,self.d_k).transpose(1,2)
        K = self.W_k(x).view(B,L,self.n_heads,self.d_k).transpose(1,2)
        V = self.W_v(x).view(B,L,self.n_heads,self.d_k).transpose(1,2)

        if past_key_values is not None:
            past_k , past_v = past_key_values
            K = torch.cat([past_k, K], dim=2)
            V = torch.cat([past_v, V], dim=2)
        present_key_values = (K, V) if use_cache else None

        score = torch.matmul(Q, K.transpose(-1,-2)) / torch.sqrt(self.d_k)
        if mask is not None:
            score = score.masked_fill(mask, float('-inf'))
        # 错误：dropout用错了对象；softmax没有制定维度
        # score = self.dropout(score)
        # attn = torch.softmax(score)
        attn = torch.softmax(score, dim=-1)
        attn = self.dropout(attn)
        output = torch.matmul(attn, V).transpose(1,2).contiguous().view(B,L,D)

        output = self.W_o(output)
        output = self.dropout(output) # 可选
        return output, present_key_values

In [ ]:
# 2026-03-09 无提示手搓
import torch
import torch.nn as nn
import math
# 遗忘：var的unbiased应该是True还是False，表达什么意义
# 好奇：torch.ones(self.normalized_shape)究竟什么样，能不能和归一化的x相乘？
class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5, elementwise_affine=True):
        super().__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape, )
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        if elementwise_affine:
            self.gamma = nn.Parameter(torch.ones(self.normalized_shape))
            self.beta = nn.Parameter(torch.zeros(self.normalized_shape))
        else:
            self.gamma = None
            self.beta = None

    def forward(self, x):
        assert x.dim() >= len(self.normalized_shape)
        dims = list(range(-len(self.normalized_shape), 0))
        mean = x.mean(dim=dims, keepdim=True)
        var = x.var(dim=dims, keepdim=True, unbiased=False)
        std = math.sqrt(var + self.eps)
        x_normalized = (x-mean) / std
        if self.elementwise_affine:
            return self.gamma * x_normalized + self.beta
        return x_normalized
    
class RMSNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5, elementwise_affine=True):
        super().__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape, )
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        if self.elementwise_affine:
            self.gamma = nn.Parameter(torch.ones(self.normalized_shape))
        else:
            self.gamma = None
    def forward(self, x):
        assert x.dim() >= len(self.normalized_shape)
        dims = list(range(-len(self.normalized_shape),0))
        # 错误，这里对RMS的定义已经是最后的修正X了
        # RMS = math.sqrt(x.pow(2).mean(dim=dims, keepdim=True) + self.eps)
        RMS = x * torch.rsqrt(x.pow(2).mean(dim=dims, keepdim=True) + self.eps)
        if self.elementwise_affine:
            # return self.gamma * (x / RMS)
            return self.gamma * RMS
        # return x / RMS
        return RMS
    
class MultiHeadAttentionWithKVCache(nn.Module):
    # 错误：bias默认应该为True，添加偏置
    # def __init__(self, d_models, n_heads=8, dropout=0.1, bias=False):
    def __init__(self, d_models, n_heads=8, dropout=0.1, bias=True):
        # 错误：竟然忘记super继承了
        super().__init__()
        assert d_models % n_heads == 0
        self.d_k = d_models // n_heads
        self.n_heads = n_heads
        self.W_q = nn.Linear(d_models, d_models, bias)
        self.W_k = nn.Linear(d_models, d_models, bias)
        self.W_v = nn.Linear(d_models, d_models, bias)
        self.W_o = nn.Linear(d_models, d_models, bias)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None, past_key_value=None, use_cache=False):
        B, L, D = x.shape
        Q = self.W_q(x).view(B,L,self.n_heads,self.d_k).transpose(1,2)
        K = self.W_k(x).view(B,L,self.n_heads,self.d_k).transpose(1,2)
        V = self.W_v(x).view(B,L,self.n_heads,self.d_k).transpose(1,2)

        if past_key_value is not None:
            past_K, past_V = past_key_value
            K = torch.cat([past_K, K], dim=2)
            V = torch.cat([past_V, V], dim=2)
        present_key_value = (K,V) if use_cache else None

        score = torch.matmul(Q, K.transpose(-1,-2)) / math.sqrt(self.d_k)
        if mask is not None:
            score = score.masked_fill(mask, float('-inf'))
        attn = torch.nn.functional.softmax(score, dim=-1)
        attn = self.dropout(attn)

        attn = torch.matmul(attn, V).transpose(1,2).contiguous().view(B,L,D)

        output = self.W_o(attn)
        return output, present_key_value
    
# 对GQA的手撕很生疏，下面的代码是没有系统学习手搓GQA的时候，根据理解来实现的，现在看来疏漏很多

class GQA(nn.Module):
    def __init__(self, d_models, n_heads=8, repeat_times=2, dropout=0.1, bias=False):
        # 整体维度，Q分的块数，K和V需要复制的次数，当前代表每一个KV要复制两次
        super().__init__()
        self.block_head = d_models // n_heads # 每一个小头的维度大小
        self.n_heads = n_heads
        self.repeat_times = repeat_times

        self.W_q = nn.Linear(d_models, self.block_head * self.n_heads, bias)
        self.W_k = nn.Linear(d_models, self.block_head * (self.n_heads // self.repeat_times), bias)
        self.W_v = nn.Linear(d_models, self.block_head * (self.n_heads // self.repeat_times), bias)
        self.W_o = nn.Linear(d_models, self.block_head * self.n_heads, bias)

        self.dropout = nn.Dropout(dropout)

    def _repeat_kv_block(self, K, V):
        # B,L,n_heads/repeat_times,block_head*(n_heads/repeat_times)
        return (torch.cat([K,K], dim=-1), torch.cat([V,V], dim=-1))
    
    def forward(self, x, mask:None, past_key_value=None, use_cache=False):
        B,L,D = x.shape
        Q = self.W_q(x).view(B,L,self.n_heads, self.block_head)
        K = self.W_k(x).view(B,L,self.n_heads//self.repeat_times, self.block_head)
        V = self.W_v(x).view(B,L,self.n_heads//self.repeat_times, self.block_head)

        if past_key_value is not None:
            past_K, past_V = past_key_value
            K = torch.cat([past_K,K],dim=1)
            V = torch.cat([past_V,V],dim=1)
        present_key_value = (K,V) if use_cache else None

        K, V = self._repeat_kv_block(K, V)
        Q = Q.transpose(1,2)
        K = K.transpose(1,2)
        V = V.transpose(1,2)

        score = torch.matmul(Q, K.transpose(-1,-2)) / math.sqrt(self.block_head*self.n_heads)
        if mask is not None:
            score = score.masked_fill(mask, float('-inf'))
        attn = torch.nn.functional.softmax(score)
        attn = self.dropout(attn)

        attn = torch.matmul(attn, V).transpose(-1,-2).contiguous().view(B,L,D)
        output = self.W_o(attn)
        return output, present_key_value
    
# 抄写一遍QGA加深印象
class GQAwithCache(nn.Module):
    def __init__(self, d_model, n_heads=8, n_kv_heads=None, dropout=0.1, bias=True):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads # 正常的Q的头数目
        self.d_k = d_model // n_heads # 每一个Q头的维度
        # 没有制定kv头数目，就用Q的头数目
        self.n_kv_heads = self.n_kv_heads if n_kv_heads is not None else n_heads
        assert n_heads % self.n_kv_heads == 0
        self.n_rep = n_heads // self.n_kv_heads # Q头是kv头的多少倍，也就是group大小

        self.W_q = nn.Linear(d_model, self.n_heads * self.d_k, bias)
        self.W_k = nn.Linear(d_model, self.n_kv_heads * self.d_k, bias)
        self.W_v = nn.Linear(d_model, self.n_kv_heads * self.d_k, bias)
        self.W_o = nn.Linear(d_model, self.n_heads * self.d_k, bias)
        self.dropout = nn.Dropout(dropout)

    # def repeat_kv(self, x: torch.Tensor, n_rep: int):
    #         """将 KV 的头数从 H_kv 扩展到 H_q (即复制 G 次)"""
    #         if n_rep == 1:
    #             return x
    #         batch, num_kv_heads, seq_len, head_dim = x.shape
    #         # 在头维度插入新维度并扩展
    #         x = x[:, :, None, :, :].expand(batch, num_kv_heads, n_rep, seq_len, head_dim)
    #         return x.reshape(batch, num_kv_heads * n_rep, seq_len, head_dim)

    def forward(self, x, past_key_value=None, use_cache=False, mask=None):
        B,L,D = x.shape
        # [B, n_heads, L, d_k]
        Q = self.W_q(x).view(B,L,self.n_heads, self.d_k).transpose(1,2)
        # [B, n_kv_heads, L, d_k]
        K = self.W_k(x).view(B,L,self.n_kv_heads, self.d_k).transpose(1,2)
        V = self.W_v(x).view(B,L,self.n_kv_heads, self.d_k).transpose(1,2)

        if past_key_value is not None:
            past_k, past_v = past_key_value
            # 在L拼接
            K = torch.cat([past_k, K], dim=2)
            V = torch.cat([past_v, V], dim=2)
        present_key_value = (K, V)
        if self.n_rep > 1:
            # K的原始维度：[B, n_kv_heads, L, d_k] K.size(2) = L
            K = K.unsqueeze(2).repeat(1,1,self.n_rep,1,1).reshape(
                B, self.n_heads, -1, self.d_k
            )
            V = V.unsqueeze(2).repeat(1,1,self.n_rep,1,1).reshape(
                B, self.n_heads, -1, self.d_k
            )
        # 也可以调用函数的写法
        # K = self.repeat_kv(K, self.group_size) # [B, Hq, Total_Len, D]
        # K = self.repeat_kv(V, self.group_size) # [B, Hq, Total_Len, D]

        score = torch.matmul(Q, K.transpose(-1,-2)) / math.sqrt(self.d_k)
        if mask is not None:
            score = score.masked_fill(mask, float('-inf'))

        attn = torch.nn.functional.softmax(score, dim=-1)
        attn = self.dropout(attn)

        output = torch.matmul(attn, V).transpose(1,2).contiguous().view(B,L,D)
        output = self.W_o(output)
        return output, present_key_value

In [ ]:
# 2026-03-12 无提示手搓
import torch
import torch.nn as nn
import torch.nn.functional as F

class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5, elementwise_affine=True):
        super().__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape, )
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        if self.elementwise_affine:
            self.gamma = nn.Parameter(torch.ones(self.normalized_shape))
            self.beta = nn.Parameter(torch.zeros(self.normalized_shape))
        else:
            self.gamma = None
            self.beta = None

    def forward(self, x):
        assert x.dim() >= len(self.normalized_shape)
        dims = list(range(-len(self.normalized_shape), 0))
        mean = x.mean(dim=dims, keepdim=True)
        # 错误1:错写为x.mean，unbiased错写为True
        # var = x.mean(dim = dims, keepdim=True, unbiased=True)
        var = x.var(dim=dims, keepdim=True, unbiased=False)
        x_normalized = (x-mean) * torch.rsqrt(var + self.eps)
        if self.normalized_shape:
            return self.gamma * x_normalized + self.beta
        return x_normalized
    
class RMSNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5, elementwise_affine=True):
        super().__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,) 
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        if self.elementwise_affine:
            self.gamma = nn.Parameter(torch.ones(self.normalized_shape))
        else:
            self.gamma = None

    def forward(self, x):
        assert x.dim() >= len(self.normalized_shape)
        dims = list(range(-len(self.normalized_shape), 0))
        RMS = x * torch.rsqrt(x.pow(2).mean(dim=dims, keepdim=True) + self.eps)
        if self.elementwise_affine:
            return self.gamma * RMS
        return RMS
    
class GQAWithCache(nn.Module):
    # 错误1:n_heads初始化为8忘记写了，bias默认为True错写为False
    def __init__(self, d_model, n_heads=8, n_kv_heads=None, dropout=0.01, bias=True):
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        # 下面这个可以优化，先定义n_kv_heads，再求rep
        # if n_kv_heads is not None:
        #     self.rep_times = n_heads // n_kv_heads
        #     self.n_kv_heads = n_kv_heads
        # else:
        #     self.rep_times = 1
        #     self.n_kv_heads = n_heads
        self.n_kv_heads = n_kv_heads if n_kv_heads is not None else n_heads
        assert self.n_heads % self.n_kv_heads == 0
        assert self.d_model % self.n_kv_heads == 0
        self.rep_times = self.n_heads // self.n_kv_heads
        self.bias = bias
        self.W_q = nn.Linear(self.d_model, self.d_k * self.n_heads, self.bias)
        self.W_k = nn.Linear(self.d_model, self.d_k * self.n_kv_heads, self.bias)
        self.W_v = nn.Linear(self.d_model, self.d_k * self.n_kv_heads, self.bias)
        self.W_o = nn.Linear(self.d_model, self.d_k * self.n_heads, self.bias)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, past_key_value=None, use_cache=False, mask=None):
        B, L, D = x.shape
        Q = self.W_q(x).view(B,L,self.n_heads, self.d_k).transpose(1,2)
        K = self.W_k(x).view(B,L,self.n_kv_heads, self.d_k).transpose(1,2)
        V = self.W_v(x).view(B,L,self.n_kv_heads, self.d_k).transpose(1,2)

        if past_key_value is not None:
            past_K, past_V = past_key_value
            K = torch.cat([past_K, K], dim=2)
            V = torch.cat([past_V, V], dim=2)
        present_key_value = (K, V) if use_cache else None

        #  开始扩张
        # 遗忘：不知道如何扩张
        # K = K.unsqueeze(2).repeat(1,1,self.rep_times,1,1).squeeze(2)
        # V = V.unsqueeze(2).repeat(1,1,self.rep_times,1,1).squeeze(2)
        # 错误2:扩张写错了，最后不是squeeze，而是reshape
        if self.rep_times > 1:
            K = K.unsqueeze(dim=2).repeat(1,1,self.rep_times,1,1).reshape(B,self.n_heads,-1,self.d_k)
            V = V.unsqueeze(dim=2).repeat(1,1,self.rep_times,1,1).reshape(B,self.n_heads,-1,self.d_k)

        score = torch.matmul(Q, K.transpose(-1,-2)) * torch.rsqrt(self.d_k)
        if mask is not None:
            score = score.masked_fill(mask, float('-inf'))
        attn = F.softmax(score, dim=-1)
        attn = self.dropout(attn)

        output= torch.matmul(attn, V).transpose(1,2).contiguous().view(B,L,D)
        output = self.W_o(output)
        # 错误3:少返回了present
        # return output
        return output, present_key_value

In [ ]:
# 2026-03-16 无提示手撕
# RMSNorm：
import torch
import torch.nn as nn
import torch.nn.functional as F

class RMSNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5, elementwise_affine=True):
        super().__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape, )
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        if self.elementwise_affine:
            self.gamma = nn.Parameter(torch.ones(self.normalized_shape))
        else:
            self.gamma = None

    def forward(self, x):
        assert x.dim() >= len(self.normalized_shape)
        dims = list(range(-len(self.normalized_shape), 0))
        RMS = x * torch.rsqrt(x.pow(2).mean(dim=dims, keepdim=True) + self.eps)
        if self.elementwise_affine:
            return self.gamma * RMS
        return RMS
    
# GQA
class GQA_cache(nn.Module):
    # 遗忘：bias到底是true还是false。 解答：这个真无所谓，随便
    def __init__(self, d_model, n_heads=8, n_kv_heads=None, dropout=0.01, bias=True):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        if n_kv_heads is None:
            n_kv_heads = n_heads
        assert n_heads % n_kv_heads == 0
        self.n_kv_heads = n_kv_heads
        self.d_k = self.d_model // self.n_heads
        self.n_rep = self.n_heads // self.n_kv_heads
        
        self.W_q = nn.Linear(self.d_model, self.d_k * self.n_heads, bias=bias)
        self.W_k = nn.Linear(self.d_model, self.d_k * self.n_kv_heads, bias=bias)
        self.W_v = nn.Linear(self.d_model, self.d_k * self.n_kv_heads, bias=bias)
        self.W_o = nn.Linear(self.d_model, self.d_k * self.n_heads, bias=bias)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, past_key_values=None, use_cache=False, mask=None):
        B, L, D = x.shape
        Q = self.W_q(x).view(B,L,self.n_heads,self.d_k).transpose(1,2)
        K = self.W_k(x).view(B,L,self.n_kv_heads,self.d_k).transpose(1,2)
        V = self.W_v(x).view(B,L,self.n_kv_heads,self.d_k).transpose(1,2)

        if past_key_values is not None:
            past_k, past_v = past_key_values
            K = torch.cat([past_k, K],dim=2)
            V = torch.cat([past_v, V],dim=2)
        present_key_values = (K,V) if use_cache else None

        # 遗忘：复制操作, B,kv_heads,L,d_k -> B,n_heads,L,d_k
        # 错误：这里直接使用repeat导致复制头错误，组之间没有对应上
        # K = K.repeat(1,self.n_rep,1,1)
        # V = V.repeat(1,self.n_rep,1,1)
        if self.n_rep > 1:
            # 使用 repeat_interleave 保证 [H0, H0, H0, H0, H1, H1...] 的顺序
            # K = K.repeat_interleave(self.n_rep, dim=1)
            # V = V.repeat_interleave(self.n_rep, dim=1)

            K = K.unsqueeze(2).expand(B,self.n_kv_heads,self.n_rep,L,D).reshape(B,self.n_heads,L,D)
            V = V.unsqueeze(2).expand(B,self.n_kv_heads,self.n_rep,L,D).reshape(B,self.n_heads,L,D)

        score = torch.matmul(Q, K.transpose(-1,-2))*torch.rsqrt(self.d_k)
        if mask is not None:
            score = score.masked_fill(mask, float('-inf'))
        attn = F.softmax(score, dim=-1)
        attn = self.dropout(attn)

        output = torch.matmul(attn, V).transpose(1,2).contiguous().view(B,L,D)
        output = self.W_o(output)
        return output, present_key_values

# 手撕快速排序，核心思想就是选择一个pivot，放到正确的位置上
def quick_sort(nums):
    if len(nums) <= 1:
        return nums
    pivot = nums[0] # 递归
    left = [x for x in nums[1:] if x < pivot]
    right = [x for x in nums[1:] if x >= pivot]
    return quick_sort(left) + pivot + quick_sort(right)

# kmeans，只知道逻辑路线，不会实现。
# 参数：k，times，过程：样本中随机选k个样本作为初始的中心样本，然后其他样本计算和这些样本的距离，归类到距离近的一组中，每一组重新找最接近平均的样本为新的中心点，如此迭代
import numpy as np
def kmeans(x, k:int, max_iters=100, tol=1e-4):
    # 输入为x,(n_samples, n_features)
    n_samples, n_features = x.shape
    indices = np.random.choice(n_samples, k, replace=False)
    centroids = x[indices]

    for i in range(max_iters):
        distances = np.linalg.norm(x[:,np.newaxis] - centroids, axis=2)
        labels = np.argmin(distances, axis=1)
        new_centriods = np.array([x[labels==j].mean(axis=0) if len(x[labels==j]) > 0 else centroids[j] for j in range(k)])
        if np.linalg.norm(new_centriods - centers) < tol:
            break
        centers = new_centriods

    return centroids, labels
